# 📊 Notebook 01 — Delivery Health Overview
## Client Delivery Intelligence — IT Services Analytics

**Business Question:** *How healthy is our overall delivery portfolio right now — and which accounts need attention first?*

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

BLUE='#1A3C6E'; RED='#C0392B'; GREEN='#27AE60'; ORANGE='#E67E22'; GRAY='#95A5A6'; LIGHT='#EEF4FB'
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white','font.family':'DejaVu Sans'})

clients    = pd.read_csv('../data/clients.csv')
projects   = pd.read_csv('../data/projects.csv')
tickets    = pd.read_csv('../data/tickets.csv')
timesheets = pd.read_csv('../data/timesheets.csv')
consultants= pd.read_csv('../data/consultants.csv')

print(f'Clients: {len(clients)} | Projects: {len(projects)} | Tickets: {len(tickets):,} | Timesheets: {len(timesheets):,}')

In [ ]:
# ── Executive KPI Dashboard ──────────────────────────────────────────────
fig = plt.figure(figsize=(20, 13), facecolor='white')
fig.text(0.5, 0.97, 'CLIENT DELIVERY INTELLIGENCE — PORTFOLIO HEALTH DASHBOARD',
         ha='center', fontsize=20, fontweight='bold', color=BLUE)
fig.text(0.5, 0.945, f'50 Accounts  |  500 Projects  |  {len(tickets):,} Tickets  |  52-Week View',
         ha='center', fontsize=11, color=GRAY)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4, top=0.93, bottom=0.05)

# KPI Cards
on_track = (projects.status=='On Track').sum()
at_risk  = (projects.status.isin(['At Risk','Delayed'])).sum()
sla_breach_rate = tickets.sla_breached.mean()*100
avg_util = timesheets.utilisation_pct.mean()
total_unbilled = timesheets.unbilled_hrs.sum()
revenue_leakage = total_unbilled * 2.5  # avg ₹2.5L per day billing rate

kpis = [
    (f'{on_track}', 'Projects On Track', GREEN, f'out of {len(projects)} total'),
    (f'{at_risk}', 'Projects At Risk / Delayed', RED, 'Requires immediate attention'),
    (f'{sla_breach_rate:.1f}%', 'SLA Breach Rate', ORANGE, 'Target: <5%'),
    (f'₹{revenue_leakage/100:.1f}Cr', 'Est. Annual Revenue Leakage', RED, 'From unbilled hours'),
]
for i,(val,label,color,sub) in enumerate(kpis):
    ax = fig.add_subplot(gs[0,i])
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.05,0.05),0.9,0.9,boxstyle='round,pad=0.05',facecolor=LIGHT,edgecolor=color,linewidth=2))
    ax.text(0.5,0.68,val,ha='center',va='center',fontsize=20,fontweight='bold',color=color)
    ax.text(0.5,0.42,label,ha='center',va='center',fontsize=9,color=BLUE,fontweight='bold',multialignment='center')
    ax.text(0.5,0.18,sub,ha='center',va='center',fontsize=8,color=GRAY,style='italic')

# Project status donut
ax2 = fig.add_subplot(gs[1,0])
status_counts = projects.status.value_counts()
status_colors = {'On Track':GREEN,'Completed':'#2980B9','At Risk':ORANGE,'Delayed':RED,'On Hold':GRAY}
colors_list = [status_colors.get(s,GRAY) for s in status_counts.index]
wedges,texts,autotexts = ax2.pie(status_counts.values,labels=status_counts.index,
    colors=colors_list,autopct='%1.0f%%',startangle=90,
    wedgeprops={'width':0.55},textprops={'fontsize':8})
for at in autotexts: at.set_fontsize(8); at.set_fontweight('bold')
ax2.set_title('Project Status\nDistribution',fontweight='bold',color=BLUE)

# SLA breach by priority
ax3 = fig.add_subplot(gs[1,1])
breach_by_priority = tickets.groupby('priority')['sla_breached'].mean()*100
breach_by_priority = breach_by_priority.sort_values(ascending=True)
colors_p = [RED if v>20 else ORANGE if v>10 else GREEN for v in breach_by_priority]
ax3.barh(breach_by_priority.index,breach_by_priority.values,color=colors_p,alpha=0.85)
ax3.axvline(5,color=RED,linestyle='--',linewidth=1.5,alpha=0.7,label='Target: 5%')
for i,(v) in enumerate(breach_by_priority.values):
    ax3.text(v+0.3,i,f'{v:.1f}%',va='center',fontsize=9,fontweight='bold')
ax3.set_xlabel('SLA Breach Rate (%)')
ax3.set_title('SLA Breach Rate\nby Ticket Priority',fontweight='bold',color=BLUE)
ax3.legend(fontsize=8)

# Cost overrun distribution
ax4 = fig.add_subplot(gs[1,2])
overrun = projects[projects.cost_overrun_pct>0]['cost_overrun_pct']
ax4.hist(overrun,bins=20,color=ORANGE,edgecolor='white',alpha=0.85)
ax4.axvline(overrun.mean(),color=RED,linestyle='--',linewidth=1.5,label=f'Avg: {overrun.mean():.1f}%')
ax4.axvline(10,color=BLUE,linestyle=':',linewidth=1.5,label='Threshold: 10%')
ax4.set_xlabel('Cost Overrun %')
ax4.set_ylabel('Number of Projects')
ax4.set_title('Cost Overrun Distribution\n(Projects with overrun)',fontweight='bold',color=BLUE)
ax4.legend(fontsize=8)

# Utilisation by grade
ax5 = fig.add_subplot(gs[1,3])
util_data = timesheets.merge(consultants[['consultant_id','grade']])
util_by_grade = util_data.groupby('grade')['utilisation_pct'].mean().sort_values()
colors_g = [RED if v>110 else ORANGE if v>95 else GREEN if v>75 else GRAY for v in util_by_grade]
ax5.barh(util_by_grade.index,util_by_grade.values,color=colors_g,alpha=0.85)
ax5.axvline(85,color=GREEN,linestyle='--',linewidth=1.5,alpha=0.8,label='Target: 85%')
ax5.axvline(100,color=RED,linestyle=':',linewidth=1.5,alpha=0.8,label='Overload: 100%')
ax5.set_xlabel('Avg Utilisation %')
ax5.set_title('Resource Utilisation\nby Grade',fontweight='bold',color=BLUE)
ax5.legend(fontsize=8)

# Ticket volume trend
ax6 = fig.add_subplot(gs[2,0:2])
weekly_tickets = tickets.groupby('week').agg(
    total=('ticket_id','count'),
    breached=('sla_breached','sum')
).reset_index()
weekly_tickets['breach_rate'] = weekly_tickets['breached']/weekly_tickets['total']*100
ax6.fill_between(weekly_tickets['week'],weekly_tickets['total'],alpha=0.15,color=BLUE)
ax6.plot(weekly_tickets['week'],weekly_tickets['total'],color=BLUE,linewidth=2,label='Total Tickets')
ax6b = ax6.twinx()
ax6b.plot(weekly_tickets['week'],weekly_tickets['breach_rate'],color=RED,linewidth=1.5,linestyle='--',label='Breach Rate %')
ax6b.set_ylabel('SLA Breach Rate (%)',color=RED)
ax6.set_xlabel('Week')
ax6.set_ylabel('Ticket Volume')
ax6.set_title('Weekly Ticket Volume vs SLA Breach Rate Trend',fontweight='bold',color=BLUE)
lines1,labels1 = ax6.get_legend_handles_labels()
lines2,labels2 = ax6b.get_legend_handles_labels()
ax6.legend(lines1+lines2,labels1+labels2,fontsize=8,loc='upper left')

# Industry breakdown
ax7 = fig.add_subplot(gs[2,2:])
proj_client = projects.merge(clients[['client_id','industry']])
industry_stats = proj_client.groupby('industry').agg(
    projects=('project_id','count'),
    avg_overrun=('cost_overrun_pct','mean'),
    at_risk=('status', lambda x: (x.isin(['At Risk','Delayed'])).sum())
).reset_index()
x = np.arange(len(industry_stats))
w = 0.35
ax7.bar(x-w/2,industry_stats['projects'],w,color=BLUE,alpha=0.85,label='Total Projects')
ax7.bar(x+w/2,industry_stats['at_risk'],w,color=RED,alpha=0.85,label='At Risk / Delayed')
ax7.set_xticks(x)
ax7.set_xticklabels([i.replace(' & ',' &\n') for i in industry_stats['industry']],fontsize=8)
ax7.set_ylabel('Number of Projects')
ax7.set_title('Project Health by Industry Vertical',fontweight='bold',color=BLUE)
ax7.legend(fontsize=9)

plt.savefig('../docs/fig01_delivery_overview.png',dpi=150,bbox_inches='tight')
plt.show()
print('Figure 01 saved.')